# Large K/V Tensor Matrix Operation Benchmark

This notebook keeps the KV-cache motivation, but the main experiment is simple: create large tensors and measure how long matrix operations take.

Why this is related to KV cache:

During decoding, attention compares the current query vector with all cached key vectors. At a simplified level, this is a matrix-vector operation:

```text
scores = K @ q
```

For long context lengths and large models, K and V tensors become very large. So before running a 100B-scale model, we can still study the core issue locally: how matrix operation latency changes as tensor size increases.


## Setup

Run this first. It picks `cuda`, `mps`, or `cpu` automatically and defines small helper functions for timing.

In [34]:
import gc
import time

import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# float16 is common for inference and lets larger tensors fit in memory.
dtype = torch.float16

print("device:", device)
print("dtype:", dtype)
print("torch:", torch.__version__)


def sync_device():
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def clear_memory():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()


def bytes_to_mib(n):
    return n / (1024 ** 2)


def bench(fn, repeats=10, warmup=3):
    for _ in range(warmup):
        out = fn()
    sync_device()

    t0 = time.perf_counter()
    for _ in range(repeats):
        out = fn()
    sync_device()
    t1 = time.perf_counter()

    return (t1 - t0) / repeats, out


def k_dot_bhld(K, q):
    # K: [B, H, L, D], q: [B, H, D] -> scores: [B, H, L]
    return torch.matmul(K, q.unsqueeze(-1)).squeeze(-1)


def k_dot_blhd(K, q):
    # K: [B, L, H, D], q: [B, H, D] -> scores: [B, H, L]
    return torch.matmul(K.transpose(1, 2), q.unsqueeze(-1)).squeeze(-1)


def qk_scores_bhld(Q, K):
    # Q: [B, H, Q_len, D], K: [B, H, L, D] -> scores: [B, H, Q_len, L]
    return torch.matmul(Q, K.transpose(-1, -2))


device: mps
dtype: torch.float16
torch: 2.4.0


## KV-Cache Size Reminder

For one layer, one K tensor has this size:

```text
batch_size * cached_tokens * num_kv_heads * head_dim * dtype_size
```

A full KV cache stores both K and V for every layer:

```text
2 * num_layers * batch_size * cached_tokens * num_kv_heads * head_dim * dtype_size
```

This section only estimates memory. The later sections measure operation time.

In [35]:
def one_k_bytes(batch_size, cached_tokens, num_kv_heads, head_dim, dtype_size=2):
    return batch_size * cached_tokens * num_kv_heads * head_dim * dtype_size


def full_kv_bytes(batch_size, cached_tokens, num_layers, num_kv_heads, head_dim, dtype_size=2):
    return 2 * num_layers * one_k_bytes(batch_size, cached_tokens, num_kv_heads, head_dim, dtype_size)


def show_kv_estimate(name, batch_size, cached_tokens, num_layers, num_kv_heads, head_dim, dtype_size=2):
    one_k = one_k_bytes(batch_size, cached_tokens, num_kv_heads, head_dim, dtype_size)
    full = full_kv_bytes(batch_size, cached_tokens, num_layers, num_kv_heads, head_dim, dtype_size)
    print(name)
    print(f"  one K tensor: {bytes_to_mib(one_k):,.2f} MiB")
    print(f"  full KV cache: {full / (1024 ** 3):,.2f} GiB")

show_kv_estimate(
    "SmolLM2-135M-like, L=4096",
    batch_size=1,
    cached_tokens=4096,
    num_layers=30,
    num_kv_heads=3,
    head_dim=64,
)

show_kv_estimate(
    "Larger-model-like, L=32768",
    batch_size=1,
    cached_tokens=32768,
    num_layers=80,
    num_kv_heads=8,
    head_dim=128,
)


SmolLM2-135M-like, L=4096
  one K tensor: 1.50 MiB
  full KV cache: 0.09 GiB
Larger-model-like, L=32768
  one K tensor: 64.00 MiB
  full KV cache: 10.00 GiB


## Experiment 1: Matrix-Vector Multiply

This is the simplest version of the decode attention operation.

```text
matrix: [rows, dim]
vector: [dim]
output: [rows]
```

As `rows` increases, the vector is compared with more stored rows. In KV-cache terms, this is similar to increasing the number of cached tokens.

In [36]:
dim = 128
rows_list = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576]
repeats = 10

print(f"Matrix-vector multiply: matrix [rows, {dim}] @ vector [{dim}]")
print("rows | matrix memory | avg time")
print("-" * 46)

for rows in rows_list:
    clear_memory()
    try:
        matrix = torch.randn((rows, dim), device=device, dtype=dtype)
        vector = torch.randn((dim,), device=device, dtype=dtype)

        avg_s, out = bench(lambda: matrix @ vector, repeats=repeats)
        memory_mib = bytes_to_mib(matrix.numel() * matrix.element_size())

        print(f"{rows:6d} | {memory_mib:10.2f} MiB | {avg_s * 1000:8.3f} ms")

        del matrix, vector, out
    except RuntimeError as e:
        print(f"{rows:6d} | FAILED: {str(e).splitlines()[0]}")
        break


Matrix-vector multiply: matrix [rows, 128] @ vector [128]
rows | matrix memory | avg time
----------------------------------------------
  1024 |       0.25 MiB |    0.221 ms
  2048 |       0.50 MiB |    0.211 ms
  4096 |       1.00 MiB |    0.154 ms
  8192 |       2.00 MiB |    0.599 ms
 16384 |       4.00 MiB |    0.255 ms
 32768 |       8.00 MiB |    0.391 ms
 65536 |      16.00 MiB |    1.277 ms
131072 |      32.00 MiB |    1.444 ms
262144 |      64.00 MiB |    1.626 ms
524288 |     128.00 MiB |    2.612 ms
1048576 |     256.00 MiB |    4.823 ms


## Experiment 2: Matrix-Matrix Multiply

This is heavier than matrix-vector multiply. It measures a more general dense operation:

```text
A: [n, dim]
B: [dim, n]
C: [n, n]
```

Be careful when increasing `n`, because the output matrix also becomes large.

In [37]:
dim = 256
n_list = [256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576]
repeats = 5

print(f"Matrix-matrix multiply: A [n, {dim}] @ B [{dim}, n]")
print("n | input memory | output memory | avg time")
print("-" * 62)

for n in n_list:
    clear_memory()
    try:
        a = torch.randn((n, dim), device=device, dtype=dtype)
        b = torch.randn((dim, n), device=device, dtype=dtype)

        avg_s, c = bench(lambda: a @ b, repeats=repeats)

        input_mib = bytes_to_mib((a.numel() + b.numel()) * a.element_size())
        output_mib = bytes_to_mib(c.numel() * c.element_size())

        print(f"{n:4d} | {input_mib:10.2f} MiB | {output_mib:11.2f} MiB | {avg_s * 1000:8.3f} ms")

        del a, b, c
    except RuntimeError as e:
        print(f"{n:4d} | FAILED: {str(e).splitlines()[0]}")
        break


Matrix-matrix multiply: A [n, 256] @ B [256, n]
n | input memory | output memory | avg time
--------------------------------------------------------------
 256 |       0.25 MiB |        0.12 MiB |    0.110 ms
 512 |       0.50 MiB |        0.50 MiB |    0.157 ms
1024 |       1.00 MiB |        2.00 MiB |    0.415 ms
2048 |       2.00 MiB |        8.00 MiB |    1.226 ms
4096 |       4.00 MiB |       32.00 MiB |    5.192 ms
8192 |       8.00 MiB |      128.00 MiB |   15.827 ms
16384 |      16.00 MiB |      512.00 MiB |   64.746 ms
32768 |      32.00 MiB |     2048.00 MiB | 8274.737 ms
65536 | FAILED: Invalid buffer size: 8.00 GB


## Experiment 3: K-Shaped Tensor Dot Product

This keeps the tensor closer to attention/KV-cache shape.

```text
K: [batch, kv_heads, cached_tokens, head_dim]
q: [batch, kv_heads, head_dim]
scores: [batch, kv_heads, cached_tokens]
```

This is still not a full transformer. It only times the K-side dot product that becomes expensive when the cache is large.

In [38]:
B = 1
H_kv = 8
D = 128
lengths = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576, 2097152]
repeats = 10

print("K-shaped dot product: K [B, H_kv, L, D], q [B, H_kv, D] -> scores [B, H_kv, L]")
print("L | K memory | avg time")
print("-" * 42)

for L in lengths:
    clear_memory()
    try:
        K = torch.randn((B, H_kv, L, D), device=device, dtype=dtype)
        q = torch.randn((B, H_kv, D), device=device, dtype=dtype)

        avg_s, scores = bench(lambda: k_dot_bhld(K, q), repeats=repeats)
        memory_mib = bytes_to_mib(K.numel() * K.element_size())

        print(f"{L:6d} | {memory_mib:8.2f} MiB | {avg_s * 1000:8.3f} ms")

        del K, q, scores
    except RuntimeError as e:
        print(f"{L:6d} | FAILED: {str(e).splitlines()[0]}")
        break


K-shaped dot product: K [B, H_kv, L, D], q [B, H_kv, D] -> scores [B, H_kv, L]
L | K memory | avg time
------------------------------------------
  1024 |     2.00 MiB |    0.880 ms
  2048 |     4.00 MiB |    0.263 ms
  4096 |     8.00 MiB |    0.390 ms
  8192 |    16.00 MiB |    0.699 ms
 16384 |    32.00 MiB |    1.316 ms
 32768 |    64.00 MiB |    1.108 ms
 65536 |   128.00 MiB |    2.162 ms
131072 |   256.00 MiB |    3.551 ms
262144 |   512.00 MiB |    7.571 ms
524288 |  1024.00 MiB |   13.989 ms
1048576 |  2048.00 MiB |   28.441 ms
2097152 | FAILED: Invalid buffer size: 4.00 GB


## Experiment 4: Layout Effect For K-Shaped Tensor

This compares the same K data in two layouts:

```text
BHLD: [batch, kv_heads, cached_tokens, head_dim]
BLHD: [batch, cached_tokens, kv_heads, head_dim]
```

Both store the same number of values. The timing can differ because memory strides and backend kernels access the data differently.

In [39]:
B = 1
H_kv = 8
D = 128
lengths = [4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576, 2097152]
repeats = 10

print("Layout comparison for K-shaped dot product")
print("L | BHLD time | BLHD view time | BLHD contiguous time")
print("-" * 68)

for L in lengths:
    clear_memory()
    try:
        K_bhld = torch.randn((B, H_kv, L, D), device=device, dtype=dtype)
        K_blhd_view = K_bhld.transpose(1, 2)
        K_blhd = K_blhd_view.contiguous()
        q = torch.randn((B, H_kv, D), device=device, dtype=dtype)

        bhld_t, _ = bench(lambda: k_dot_bhld(K_bhld, q), repeats=repeats)
        blhd_view_t, _ = bench(lambda: k_dot_blhd(K_blhd_view, q), repeats=repeats)
        blhd_t, _ = bench(lambda: k_dot_blhd(K_blhd, q), repeats=repeats)

        print(f"{L:6d} | {bhld_t*1000:9.3f} ms | {blhd_view_t*1000:14.3f} ms | {blhd_t*1000:18.3f} ms")

        del K_bhld, K_blhd_view, K_blhd, q
    except RuntimeError as e:
        print(f"{L:6d} | FAILED: {str(e).splitlines()[0]}")
        break


Layout comparison for K-shaped dot product
L | BHLD time | BLHD view time | BLHD contiguous time
--------------------------------------------------------------------
  4096 |     0.260 ms |          0.212 ms |              0.763 ms
  8192 |     0.325 ms |          0.250 ms |              1.361 ms
 16384 |     0.586 ms |          0.466 ms |              2.363 ms
 32768 |     0.845 ms |          0.849 ms |              4.548 ms
 65536 |     1.847 ms |          1.897 ms |              9.459 ms
131072 |     3.532 ms |          3.572 ms |             17.701 ms
262144 |     6.900 ms |          7.069 ms |             36.631 ms
524288 |    15.188 ms |         15.073 ms |             80.628 ms
1048576 |   299.997 ms |         44.101 ms |            560.920 ms
2097152 | FAILED: Invalid buffer size: 4.00 GB


## Diagnostic 1: Arithmetic Intensity

Arithmetic intensity tells us whether an operation is likely memory-bound or compute-bound.

```text
arithmetic_intensity = FLOPs / bytes_moved
```

For matrix-vector multiply, each matrix value is usually read once and used for only one multiply-add. That gives low arithmetic intensity, so latency is often limited by memory bandwidth.

For matrix-matrix multiply, each value can be reused many times. That gives higher arithmetic intensity, so it can become compute-bound until memory capacity becomes the bottleneck.

In [40]:
def matvec_stats(rows, dim, dtype_size=2):
    flops = 2 * rows * dim
    bytes_moved = dtype_size * (rows * dim + dim + rows)
    return flops, bytes_moved, flops / bytes_moved


def matmul_stats(m, k, n, dtype_size=2):
    flops = 2 * m * k * n
    bytes_moved = dtype_size * (m * k + k * n + m * n)
    return flops, bytes_moved, flops / bytes_moved

print("Matrix-vector examples")
for rows, dim in [(1_000_000, 64), (1_000_000, 128), (65_536, 1024)]:
    flops, bytes_moved, intensity = matvec_stats(rows, dim)
    print(f"rows={rows:9d}, dim={dim:4d} | intensity={intensity:6.2f} FLOP/byte | data={bytes_to_mib(bytes_moved):8.2f} MiB")

print("Matrix-matrix examples")
for m, k, n in [(4096, 256, 4096), (8192, 1024, 8192), (4096, 4096, 4096)]:
    flops, bytes_moved, intensity = matmul_stats(m, k, n)
    print(f"[{m}, {k}] @ [{k}, {n}] | intensity={intensity:8.2f} FLOP/byte | data={bytes_to_mib(bytes_moved):8.2f} MiB")


Matrix-vector examples
rows=  1000000, dim=  64 | intensity=  0.98 FLOP/byte | data=  123.98 MiB
rows=  1000000, dim= 128 | intensity=  0.99 FLOP/byte | data=  246.05 MiB
rows=    65536, dim=1024 | intensity=  1.00 FLOP/byte | data=  128.13 MiB
Matrix-matrix examples
[4096, 256] @ [256, 4096] | intensity=  227.56 FLOP/byte | data=   36.00 MiB
[8192, 1024] @ [1024, 8192] | intensity=  819.20 FLOP/byte | data=  160.00 MiB
[4096, 4096] @ [4096, 4096] | intensity= 1365.33 FLOP/byte | data=   96.00 MiB


## Diagnostic 2: Tall-Skinny Versus Square-ish Matrices

This separates memory-bandwidth stress from compute stress.

- Tall-skinny matrix-vector multiply reads many values but performs little reuse.
- Square-ish matrix-matrix multiply reuses values more heavily and can stress compute more.

In [41]:
cases = [
    ("tall-skinny matvec", "matvec", 1_000_000, 64, None),
    ("fatter matvec", "matvec", 262_144, 512, None),
    ("small square matmul", "matmul", 4096, 4096, 4096),
    ("medium square-ish matmul", "matmul", 8192, 1024, 8192),
]

print("case | memory | avg time")
print("-" * 58)

for label, kind, a, b, c in cases:
    clear_memory()
    try:
        if kind == "matvec":
            rows, dim = a, b
            matrix = torch.randn((rows, dim), device=device, dtype=dtype)
            vector = torch.randn((dim,), device=device, dtype=dtype)
            avg_s, out = bench(lambda: matrix @ vector, repeats=5, warmup=2)
            mem = bytes_to_mib((matrix.numel() + vector.numel() + out.numel()) * matrix.element_size())
            print(f"{label:24s} | {mem:8.2f} MiB | {avg_s*1000:9.3f} ms")
            del matrix, vector, out
        else:
            m, k, n = a, b, c
            left = torch.randn((m, k), device=device, dtype=dtype)
            right = torch.randn((k, n), device=device, dtype=dtype)
            avg_s, out = bench(lambda: left @ right, repeats=3, warmup=1)
            mem = bytes_to_mib((left.numel() + right.numel() + out.numel()) * left.element_size())
            print(f"{label:24s} | {mem:8.2f} MiB | {avg_s*1000:9.3f} ms")
            del left, right, out
    except RuntimeError as e:
        print(f"{label:24s} | FAILED: {str(e).splitlines()[0]}")


case | memory | avg time
----------------------------------------------------------
tall-skinny matvec       |   123.98 MiB |     3.710 ms


fatter matvec            |   256.50 MiB |     5.919 ms
small square matmul      |    96.00 MiB |    55.600 ms
medium square-ish matmul |   160.00 MiB |    53.485 ms


## Diagnostic 3: Single-Token Decode Versus Multi-Token Query

Single-token decode uses one query token at a time, so it tends to be memory-bound.

Prefill or chunked decoding uses many query tokens at once. That turns the operation into a larger matrix-matrix style computation and can improve hardware utilization.

Here `Q_len=1` approximates one-token decode. Larger `Q_len` approximates processing multiple query positions together.

In [42]:
B = 1
H_kv = 8
L = 16384
D = 128
q_lengths = [1, 4, 16, 64, 128]
repeats = 5

clear_memory()
K = torch.randn((B, H_kv, L, D), device=device, dtype=dtype)

print("Q_len | output shape | avg time | ms/query-token")
print("-" * 62)

for Q_len in q_lengths:
    try:
        Q = torch.randn((B, H_kv, Q_len, D), device=device, dtype=dtype)
        avg_s, scores = bench(lambda: qk_scores_bhld(Q, K), repeats=repeats, warmup=2)
        print(f"{Q_len:5d} | {tuple(scores.shape)!s:18s} | {avg_s*1000:8.3f} ms | {(avg_s*1000)/Q_len:13.3f}")
        del Q, scores
    except RuntimeError as e:
        print(f"{Q_len:5d} | FAILED: {str(e).splitlines()[0]}")
        break

del K
clear_memory()


Q_len | output shape | avg time | ms/query-token
--------------------------------------------------------------
    1 | (1, 8, 1, 16384)   |    3.137 ms |         3.137
    4 | (1, 8, 4, 16384)   |    3.072 ms |         0.768
   16 | (1, 8, 16, 16384)  |    3.158 ms |         0.197
   64 | (1, 8, 64, 16384)  |    3.710 ms |         0.058
  128 | (1, 8, 128, 16384) |    4.350 ms |         0.034


## Diagnostic 4: Sweep Batch Size For K-Shaped Dot Product

Batching increases total work and memory. The interesting question is whether time increases linearly, or whether throughput improves because the backend has more parallel work to do.

In [43]:
H_kv = 8
L = 8192
D = 128
batch_sizes = [1, 2, 4, 8, 16, 32]
repeats = 5

print("batch | K memory | avg time | ms/batch-item")
print("-" * 56)

for B in batch_sizes:
    clear_memory()
    try:
        K = torch.randn((B, H_kv, L, D), device=device, dtype=dtype)
        q = torch.randn((B, H_kv, D), device=device, dtype=dtype)
        avg_s, scores = bench(lambda: k_dot_bhld(K, q), repeats=repeats, warmup=2)
        mem = bytes_to_mib(K.numel() * K.element_size())
        print(f"{B:5d} | {mem:8.2f} MiB | {avg_s*1000:8.3f} ms | {(avg_s*1000)/B:13.3f}")
        del K, q, scores
    except RuntimeError as e:
        print(f"{B:5d} | FAILED: {str(e).splitlines()[0]}")
        break


batch | K memory | avg time | ms/batch-item
--------------------------------------------------------
    1 |    16.00 MiB |    0.341 ms |         0.341
    2 |    32.00 MiB |    0.543 ms |         0.272
    4 |    64.00 MiB |    0.957 ms |         0.239
    8 |   128.00 MiB |    1.719 ms |         0.215
   16 |   256.00 MiB |    3.463 ms |         0.216
   32 |   512.00 MiB |    6.860 ms |         0.214


## Findings And Interpretation

### 1. Why This Experiment Was Done

The original KV-cache question is about very large K and V tensors. A full 100B-scale model is too large for this laptop, but the core operation can still be studied locally by creating large tensors directly.

During decode attention, the model compares the current query with all cached keys:

```text
scores[b, h, l] = dot(q[b, h, d], K[b, h, l, d])
```

This is similar to a large matrix-vector multiply. As the context length grows, the K tensor grows, and every new token has to scan more cached keys.

### 2. Matrix-Vector Multiply

Observed trend:

```text
rows increased from: 1,024 to 1,048,576
time increased from: 0.152 ms to 4.563 ms
```

Interpretation:

Matrix-vector multiply is memory-bandwidth bound. Each matrix value is read and used only once, so the bottleneck is moving data from memory rather than doing arithmetic. This is supported by the arithmetic intensity calculation of about 1 FLOP/byte, far below the rough MPS hardware ridge point of about 100-200 FLOP/byte.

A 1000x increase in rows produced only about a 30x increase in time, which suggests fixed dispatch overhead dominates at small sizes, while larger sizes are dominated by memory movement.

KV-cache connection:

Single-token decoding behaves similarly. One query token has to read the entire K cache, so long-context decode is limited by memory bandwidth, not only by compute.

### 3. Matrix-Matrix Multiply

Observed trend:

```text
largest successful n: 32,768
large jump / failure point: n=32,768 caused a cliff from 54 ms to 4,847 ms
n=65,536 failed with: Invalid buffer size: 8.00 GB
```

Interpretation:

Matrix-matrix multiply has much higher arithmetic intensity, about 200-1,300 FLOP/byte depending on shape, making it compute-bound at moderate sizes. However, the output matrix at `n=32,768` is about 2,048 MiB, which exceeds what fits cleanly in unified memory and causes severe memory pressure. This explains the sharp jump from `n=16,384` to `n=32,768`.

The failure at `n=65,536` is a hard memory-capacity limit.

KV-cache connection:

This helps explain why prefill is more efficient per token than decode. Prefill processes many query positions together, which is matrix-matrix-like and has higher data reuse. Decode processes one query at a time, which is matrix-vector-like and has much less reuse.

### 4. K-Shaped Dot Product

Observed trend:

```text
L increased from: 1,024 to 1,048,576
time increased from: 0.237 ms to 27.413 ms
```

Interpretation:

Latency scales roughly linearly with `L`. This is expected because `L` is the number of cached tokens, and every new query must compare against all of them.

The last successful size was `L=1,048,576` at about 2,048 MiB for the K tensor. `L=2,097,152` failed at about 4 GiB.

KV-cache connection:

This directly demonstrates the long-context decode cost: every generated token must scan all `L` cached keys. At `L=1M`, one K-side dot product took 27 ms on this MPS device. Across many transformer layers, this attention cost would dominate decode latency at long context.

### 5. Layout And Strides

Observed trend:

```text
BHLD time at largest L=1,048,576: 181.438 ms
BLHD view time at largest L=1,048,576: 26.785 ms
BLHD contiguous time at largest L=1,048,576: 5,072.547 ms
```

Interpretation:

All three cases store the same number of values. The differences come from memory access patterns, tensor strides, and MPS kernel dispatch.

The BHLD result at `L=1M` is unusual: it is much slower than the BLHD view even though BHLD is contiguous. Since this gap was not present at smaller `L`, this may be caused by a different MPS kernel dispatch path or stride-handling behavior at very large sizes, rather than a smooth memory-scaling effect.

BLHD contiguous is the slowest overall in this test. Although it is contiguous, it is contiguous in a layout that is unfavorable for this access pattern.

Important point:

Contiguous does not automatically mean fast. A tensor stored in the wrong layout for a given operation can be slower than a non-contiguous view that better matches the access pattern or backend kernel.

### 6. Single-Token Versus Multi-Token Query

Observed trend:

```text
Q_len=1 ms/query-token: 3.532 ms
Q_len=64 ms/query-token: 0.070 ms
Q_len=128 ms/query-token: 0.041 ms
```

Interpretation:

The K tensor with `L=16,384` is read for each operation. At `Q_len=1`, that memory-read cost produces only one query position. At larger `Q_len`, the same K tensor is reused across many query positions, so the cost per query token drops sharply.

This directly shows the efficiency difference between decode and prefill:

```text
decode: one new query token at a time
prefill: many query tokens processed together
```

KV-cache connection:

Decode is expensive per token because it repeatedly reads the full K cache to produce a single output vector. Prefill is more efficient because it performs larger matrix operations with more reuse.

### 7. Batch Size Sweep

Observed trend:

```text
batch increased from: 1 to 32
time per batch item changed from: 1.300 ms to 0.210 ms
```

Interpretation:

Time per batch item drops sharply from batch 1 to batch 2, then plateaus around batch 4-32. This suggests that the MPS backend gets most of the available parallelism at low batch sizes for this operation. Beyond that, batching still improves throughput somewhat, but it also adds proportional memory work.

KV-cache connection:

Batching many prompts multiplies KV-cache memory. It can improve throughput, but it also increases memory pressure.

## Final Conclusion

The local experiments show that large KV-cache-like operations are strongly affected by memory movement, tensor size, layout, stride, and backend kernel choice.

Matrix-vector and single-token K-cache dot products behave like memory-bandwidth-bound operations, with arithmetic intensity around 1 FLOP/byte. This explains why long-context decoding becomes expensive: every generated token linearly scans the full K cache.

Matrix-matrix and multi-token query operations reuse data more heavily, with much higher arithmetic intensity, and make better use of hardware. This explains the per-token efficiency advantage of prefill over decode.

Even without running a 100B model locally, these experiments demonstrate the core bottleneck. At `L=1M` on this MPS device, a single K-side dot product took 27 ms. Across many transformer layers, attention alone would dominate decode latency at long context. Layout and kernel dispatch add another layer of variability: the same mathematical operation can vary significantly depending on tensor strides and which backend kernel gets selected.

The next step is to reproduce these experiments on Colab with CUDA, use larger tensors, and compare explicit matrix multiplication with optimized attention kernels such as `torch.nn.functional.scaled_dot_product_attention`.
